# Week 5, Lab 01: LoRA Catalog Normalization

**Cordwell Home and Hardware** ingests product data from 300 suppliers. Every supplier sends free-text product blurbs in a different style. Downstream systems need a strict JSON attribute record. Prompting a small base model gets partial compliance and inconsistent field naming. Your task: fine-tune a LoRA adapter that emits the schema reliably, then prove it with real numbers from your own run.

## Objectives

By the end of this lab you can:

1. Baseline a frozen model on a held-out set and defend why the baseline comes first
2. Audit training data for quality defects and explain why data quality dominates outcome
3. Configure a LoraConfig and interpret its trainable parameter report
4. Run a LoRA fine-tune with trl and read its training log
5. Re-measure on the same held-out set and separate schema-valid from correct

## Time budget (about 3 hours core, 4 with stretch goals)

| Part | Task | Minutes |
|---|---|---|
| Setup | Environment, data, worked example | 10 |
| A | Baseline the frozen model on 50 records | 25 |
| B | Find and fix the 3 planted data quality defects | 45 |
| C | Build LoraConfig, report trainable params | 25 |
| D | Run the fine-tune | 35 |
| E | Re-measure on the same 50 records | 25 |
| F | Stretch: rerun at r=4, compare | 30 |
| G | Stretch: capability regression check | 20 |

## Backend selector

Set the `TRAIN_BACKEND` environment variable **before** launching Jupyter. Explicit selection raises a clear error rather than silently switching.

| Value | Path | Notes |
|---|---|---|
| `peft_mps` (default) | Real fine-tune, peft and trl on MPS or CPU | The transferable API |
| `mlx` | Apple-native command line path | See APPENDIX_MLX.md |
| `offline` | Tiny built-in stand-in model, deterministic outputs | Always completes every part, no downloads |

What is pre-written for you: data loading, the JSON schema validator, the metric computation, the evaluation loop, and plotting (all in `lab_support.py`). You write the LoraConfig, the SFTConfig, the training call, and the data audit.

Hints ship in two tiers. `HINTS.md` names the approach and sketches structure. `HINTS_DETAILED.md` shows the working core with commentary. Pick one tier per task; reading both wastes time.

In [ ]:
%pip install -r requirements.txt

## Worked target output

This is what done looks like. One real record, shown in full, before you write any code. The blurb goes in, the strict schema comes out. Every metric in this lab counts how often the model produces the right column against inputs like the left column.

Input, a supplier blurb verbatim:

```text
HEAVY DUTY 3/4in BRASS ball valve, full port, 600 WOG, FNPT end type, sold 12/case. MFG#BV-3475.
```

Target, the strict schema output. The whole response must be exactly one JSON object, no prose, no markdown fences:

```json
{
  "sku_mfg": "BV-3475",
  "category": "plumbing.valves.ball",
  "material": "brass",
  "size_nominal_in": 0.75,
  "pack_qty": 12,
  "attributes": {
    "port": "full",
    "pressure_rating": "600 WOG",
    "end_type": "FNPT"
  }
}
```

Seeing the target shape clarifies the contract. It does not reveal the solution logic, which is why it is not a hint.

In [ ]:
# Setup. Pre-written. Run as-is.
import json
import lab_support as ls
from lab_support import check

ls.backend_banner()

In [ ]:
# Build the datasets (deterministic from seed 42) and show the worked
# example as live data. Pre-written. Run as-is.
summary = ls.build_datasets()
print("data:", summary)

train_records = ls.load_jsonl(ls.TRAIN_FILE)
eval_records = ls.load_jsonl(ls.EVAL_FILE)

example = eval_records[0]
print("\nBLURB:", ls.blurb_of(example))
print("\nTARGET:")
print(json.dumps(ls.gold_of(example), indent=2))

check("setup: 450 training records", lambda: len(train_records) == 450)
check("setup: 50 eval records", lambda: len(eval_records) == 50)
check("setup: backend selected", lambda: ls.TRAIN_BACKEND in ("peft_mps", "offline"))

## Part A: Baseline the frozen model (25 min)

You cannot claim improvement without a before. Part A measures the frozen base model on all 50 held-out records with the exact metric the downstream system cares about: schema-valid rate, plus category accuracy and per-field exact match.

The plumbing exists. `ls.generate_for_eval` produces one raw output string per eval record. `ls.evaluate_outputs` scores a list of output strings against the gold records. Your job is to compose them into `run_evaluation`, which you will reuse verbatim in Parts E and F. That reuse is the point: one evaluation function, run before and after, same records, same scoring.

Expected result in offline mode: schema-valid rate 34 percent. In live mode expect roughly 20 to 45 percent; the exact number depends on the base model build. Either way it will be bad. That is the before picture.

In [ ]:
def run_evaluation(model, tokenizer, eval_records, phase):
    """Generate one output per eval record, then score the outputs."""
    outputs = ls.generate_for_eval(model, tokenizer, eval_records, phase)
    return ls.evaluate_outputs(outputs, eval_records)

In [ ]:
# Part A driver. Pre-written. Run as-is.
base_model, tokenizer = ls.load_base_model()

baseline_results = None
try:
    baseline_results = run_evaluation(base_model, tokenizer,
                                      eval_records, "baseline")
    ls.print_results("Part A: frozen baseline", baseline_results)
except NotImplementedError:
    print("run_evaluation not implemented yet")

check("A1: results dict has the four metric keys",
      lambda: {"schema_valid_rate", "category_accuracy",
               "per_field", "failures"} <= set(ls.need(baseline_results)))

Look at the `failures` list before moving on. Prose preambles, markdown fences, single quotes, renamed fields: these are the exact behaviors the fine-tune must remove. The model mostly knows the content. It cannot hold the format. That is the definition of a behavior problem, and behavior problems are what LoRA is for.

## Part B: Audit the training data (45 min)

Three defect types were planted in the 450 training records. Data quality dominates outcome: a fine-tune faithfully learns whatever you feed it, including the garbage. Find all three types before any training happens.

The defect types, in the order a pipeline usually meets them:

1. **Invalid JSON.** The assistant label does not parse. `json.loads` rejects it.
2. **Field drift.** The label parses, but a key does not match the schema. One supplier feed renamed `sku_mfg`.
3. **Unit mismatch.** The label parses and passes the schema check, but `size_nominal_in` disagrees with the size stated in the blurb text. Somebody exported millimeters into an inches column.

Defect 3 is the important one. It is invisible to the schema validator. Schema-valid is not correct, and this lab makes you meet that fact in the data before you meet it again in the model outputs in Part E.

Provided tools: `ls.validate_output(text)` returns `(ok, problems)` where each problem is a human-readable string. `ls.extract_size_from_blurb(blurb)` returns the nominal size in inches stated in the blurb, or None. `ls.blurb_of(record)` returns the blurb.

In [ ]:
def audit_training_data(records):
    """Identify the three planted defect types."""
    found = {"invalid_json": [], "field_drift": [], "unit_mismatch": []}
    for rec in records:
        label = rec["messages"][2]["content"]
        ok, problems = ls.validate_output(label)
        if not ok:
            if any("not valid JSON" in p for p in problems):
                found["invalid_json"].append(rec["rid"])
            else:
                found["field_drift"].append(rec["rid"])
            continue
        gold = json.loads(label)
        stated = ls.extract_size_from_blurb(ls.blurb_of(rec))
        labeled = gold["size_nominal_in"]
        if (labeled is not None and stated is not None
                and abs(labeled - stated) > 0.01):
            found["unit_mismatch"].append(rec["rid"])
    return {k: sorted(v) for k, v in found.items()}

In [ ]:
# Part B audit driver. Pre-written. Run as-is.
import hashlib

def _digest(rids):
    return hashlib.sha256(",".join(sorted(rids)).encode()).hexdigest()[:16]

audit = None
try:
    audit = audit_training_data(train_records)
    for k, v in audit.items():
        print(f"{k}: {len(v)} records, first three: {v[:3]}")
except NotImplementedError:
    print("audit_training_data not implemented yet")

check("B1: found all 8 invalid JSON records",
      lambda: _digest(ls.need(audit)["invalid_json"]) == "27bc9671b09f76dd")
check("B2: found all 10 field drift records",
      lambda: _digest(ls.need(audit)["field_drift"]) == "9fc14bf511718964")
check("B3: found all 6 unit mismatch records",
      lambda: _digest(ls.need(audit)["unit_mismatch"]) == "dcf7ac32a92c637d")

### Repair or drop

The cleaning policy, argued once here so the code does not have to argue it:

- **Field drift: repair.** The rename is mechanical and lossless. `mfg_sku` becomes `sku_mfg`, everything else is intact. Dropping 10 good labels over a key name wastes signal.
- **Unit mismatch: repair.** The blurb states the true size and the extractor reads it reliably. Replace the labeled value with the blurb value.
- **Invalid JSON: drop.** Repairing the quotes would be easy, but a label that failed to serialize correctly has an untrusted provenance; you do not know what else the exporter mangled. Dropping 8 of 450 is cheaper than debugging a polluted fine-tune later.

Expected result: 442 clean records.

In [ ]:
def clean_training_data(records, audit):
    """Apply the repair-or-drop policy above."""
    drop = set(audit["invalid_json"])
    drift = set(audit["field_drift"])
    unit = set(audit["unit_mismatch"])
    cleaned = []
    for rec in records:
        if rec["rid"] in drop:
            continue
        rec = json.loads(json.dumps(rec))  # deep copy, no mutation
        label = rec["messages"][2]["content"]
        if rec["rid"] in drift:
            obj = json.loads(label)
            obj = {("sku_mfg" if k == "mfg_sku" else k): v
                   for k, v in obj.items()}
            obj = {k: obj[k] for k in ls.REQUIRED_FIELDS}
            rec["messages"][2]["content"] = json.dumps(obj)
        elif rec["rid"] in unit:
            obj = json.loads(label)
            obj["size_nominal_in"] = ls.extract_size_from_blurb(
                ls.blurb_of(rec))
            rec["messages"][2]["content"] = json.dumps(obj)
        cleaned.append(rec)
    return cleaned

In [ ]:
# Part B clean driver. Pre-written. Run as-is.
clean_records = None
try:
    clean_records = clean_training_data(train_records, ls.need(audit))
    print(f"clean records: {len(clean_records)}")
except NotImplementedError:
    print("clean_training_data not implemented yet")

check("B4: 442 records survive cleaning",
      lambda: len(ls.need(clean_records)) == 442)
check("B5: every surviving label passes the schema check",
      lambda: all(ls.validate_output(r["messages"][2]["content"])[0]
                  for r in ls.need(clean_records)))
check("B6: no unit mismatches remain",
      lambda: not audit_training_data(ls.need(clean_records))["unit_mismatch"])

if clean_records:
    with open(ls.CLEAN_FILE, "w") as f:
        for rec in clean_records:
            f.write(json.dumps(rec) + "\n")
    print(f"wrote {ls.CLEAN_FILE}")

## Part C: Build the LoraConfig (25 min)

Now the config from the module, applied. Five arguments, each mapped to a concept:

| Argument | Concept | Value for this task |
|---|---|---|
| `r` | Capacity: the rank of the A and B matrices | 16 |
| `lora_alpha` | Influence: the ratio alpha over r is what matters | 32 |
| `target_modules` | Reach: which projections get adapters | `q_proj`, `v_proj` |
| `lora_dropout` | Regularization inside the adapter | 0.05 |
| `task_type` | Tells PEFT which wrapper to build | causal LM |

Q and V alone is the default for format and style, which is exactly this task. Set `bias` to `"none"`.

After wrapping, `print_trainable_parameters` is your receipt. On the real base model (a 0.36B GQA architecture) expect roughly 1.6M trainable parameters, about 0.45 percent of the base. The offline stand-in is tiny, so its percentage reads higher, near 4 percent; the check accepts both because the invariant that matters is trainable is a small fraction of total and the base is frozen.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

def build_lora_config():
    """Return the LoraConfig specified in the table above."""
    return LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

In [ ]:
# Part C driver. Pre-written. Run as-is.
# Reload the base fresh so re-running this cell never double-wraps it.
base_model, tokenizer = ls.load_base_model()

lora_config = None
peft_model = None
try:
    lora_config = build_lora_config()
    peft_model = get_peft_model(base_model, lora_config)
    peft_model.print_trainable_parameters()
except NotImplementedError:
    print("build_lora_config not implemented yet")

check("C1: r=16 and lora_alpha=32",
      lambda: ls.need(lora_config).r == 16 and lora_config.lora_alpha == 32)
check("C2: adapters target q_proj and v_proj",
      lambda: set(ls.need(lora_config).target_modules) == {"q_proj", "v_proj"})
check("C3: task type is causal LM",
      lambda: ls.need(lora_config).task_type == TaskType.CAUSAL_LM)
check("C4: trainable parameters are a small nonzero fraction",
      lambda: (lambda t, a: 0 < t / a < 0.05)(
          *ls.need(peft_model).get_nb_trainable_parameters()))

Sanity math before moving on. The scaling factor is alpha over r, so 32 over 16 gives 2.0. Doubling both to r=32, alpha=64 leaves the scaling unchanged and doubles capacity. That ratio, not either value alone, is what the forward pass sees.

## Part D: Run the fine-tune (35 min)

The training configuration and the two-line launch. In live mode this trains for 3 epochs and takes roughly 10 to 25 minutes on an M-series Mac; start it, then join the group discussion. In offline mode `EPOCHS` is 1 and the run takes seconds against the stand-in model.

Configuration notes worth internalizing:

- `learning_rate=2e-4` is roughly 10x a full fine-tune rate. You train far fewer parameters, so the signal must be stronger per step.
- `gradient_accumulation_steps=4` with batch 4 gives an effective batch of 16 without the memory of 16.
- `eval_strategy="epoch"` runs validation each epoch. The argument was renamed from `evaluation_strategy`; tutorials showing the old name are stale.
- `max_length=512` is the token budget per example. This dataset fits comfortably.
- `bf16` only on MPS. bf16 needs macOS 14 or newer; the conditional expression handles machines without it.
- `use_cpu=True` only on machines with no accelerator at all. Without it, current transformers refuses to configure a run on a pure CPU box. Harmless and False on your Macs, where MPS counts as the accelerator.
- `report_to=[]` keeps third-party experiment loggers out of the lab.

In [ ]:
# Pre-written: dataset load and split. load_dataset returns ONE split;
# the validation split is made explicitly, seeded for reproducibility.
import os
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

EPOCHS = 1 if ls.TRAIN_BACKEND == "offline" else 8

splits = None
if os.path.exists(ls.CLEAN_FILE):
    raw = load_dataset("json", data_files=ls.CLEAN_FILE, split="train")
    splits = raw.train_test_split(test_size=0.1, seed=42)
    print(f"train {len(splits['train'])}, validation {len(splits['test'])}")
else:
    print("Complete Part B first: the clean training file does not exist yet")

In [ ]:
def build_training_args():
    """Return the SFTConfig for this run."""
    return SFTConfig(
        output_dir="./lora_checkpoints",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=10,
        save_strategy="no",
        eval_strategy="epoch",
        max_length=512,
        bf16=(ls.DEVICE == "mps"),
        use_cpu=(ls.DEVICE == "cpu"),
        report_to=[],
    )


def run_training(peft_model, training_args, splits, tokenizer):
    """Construct the SFTTrainer and launch training."""
    trainer = SFTTrainer(
        model=peft_model,
        args=training_args,
        train_dataset=splits["train"],
        eval_dataset=splits["test"],
        processing_class=tokenizer,
    )
    trainer.train()
    return trainer

In [ ]:
# Part D driver. Pre-written. Run as-is.
training_args = None
trainer = None
try:
    training_args = build_training_args()
    trainer = run_training(ls.need(peft_model), training_args,
                           ls.need(splits), tokenizer)
except NotImplementedError:
    print("Part D not implemented yet")

check("D1: learning rate in the LoRA range, eval each epoch, budget 512",
      lambda: 1e-4 <= ls.need(training_args).learning_rate <= 5e-4
              and training_args.eval_strategy == "epoch"
              and training_args.max_length == 512
              and training_args.num_train_epochs == EPOCHS)
check("D2: effective batch size is 16",
      lambda: ls.need(training_args).per_device_train_batch_size
              * training_args.gradient_accumulation_steps == 16)
check("D3: training ran and logged a final train loss",
      lambda: any("train_loss" in e
                  for e in ls.need(trainer).state.log_history))

In [ ]:
# Pre-written: save the adapter and show the loss trajectory.
import os

if trainer is not None:
    trainer.model.save_pretrained("./adapters/catalog_r16")
    print("adapter saved:", sorted(os.listdir("./adapters/catalog_r16")))
    print("\nloss trajectory:")
    for entry in trainer.state.log_history:
        if "loss" in entry:
            print(f"  step {entry.get('step'):>4}  "
                  f"train loss {entry['loss']:.3f}")
        elif "eval_loss" in entry:
            print(f"  epoch {entry.get('epoch'):>4}  "
                  f"eval loss  {entry['eval_loss']:.3f}")

While training runs, read the log the way the module taught. Training loss falling and validation loss falling with it is a healthy run. Training falling while validation rises is overfitting: more dropout or fewer epochs. Both flat and high is almost always a data format problem, and you would inspect the messages column first, not the hyperparameters.

Note what the adapter on disk is: `adapter_config.json` plus `adapter_model.safetensors`, a few megabytes. The base model is not in there. That file size is the whole deployment story from Segment 5: swap a file, not a model.

## Part E: Re-measure on the same 50 records (25 min)

Same records, same scoring, same function you wrote in Part A. The only change is the model. Anything else would be a rigged demo.

Expected in offline mode: schema-valid rate 98 percent, category accuracy 88 percent. In live mode expect a large jump; confirm your own numbers. Then look closely at the gap between the two headline metrics, because it is the day's second lesson in disguise.

In [ ]:
def evaluate_tuned(trainer, tokenizer, eval_records):
    """Re-run the Part A evaluation against the trained model."""
    return run_evaluation(trainer.model, tokenizer,
                          eval_records, "tuned_r16")

In [ ]:
# Part E driver. Pre-written. Run as-is.
tuned_results = None
try:
    tuned_results = evaluate_tuned(ls.need(trainer), tokenizer, eval_records)
    ls.print_results("Part E: tuned adapter, r=16", tuned_results)
except NotImplementedError:
    print("evaluate_tuned not implemented yet")

check("E1: tuned schema-valid rate beats baseline by 30 points or more",
      lambda: ls.need(tuned_results)["schema_valid_rate"]
              >= ls.need(baseline_results)["schema_valid_rate"] + 0.30)
check("E2: tuned per-field breakdown present for all six fields",
      lambda: set(ls.need(tuned_results)["per_field"]) == set(ls.REQUIRED_FIELDS))

In [ ]:
# Pre-written: the before-and-after picture.
if baseline_results and tuned_results:
    results = {"baseline": baseline_results, "tuned r=16": tuned_results}
    ls.plot_before_after(results)
    ls.plot_field_breakdown(results)

Read the gap. Schema-valid rate 98 percent, category accuracy 88 percent in offline mode: five records emit perfectly valid JSON with the wrong taxonomy leaf. The validator waves them through, downstream systems mis-shelve the product. Schema-valid is not correct, in the model outputs now exactly as it was in the training data in Part B. Headline metrics need a per-field breakdown behind them, every time.

Loss is your build passing. These metrics are your tests passing. You now have both, plus the before picture to prove the delta. That triple is the minimum honest claim of improvement.

## Part F, stretch: capacity in action (30 min)

Rerun the whole pipeline at r=4, alpha=8. Same ratio, quarter capacity. The module's claim: rank buys capacity, and underfitting shows up as the metric plateauing below the higher rank. Verify it with your own numbers instead of trusting the slide.

In offline mode the r=4 run lands at 90 percent schema-valid against 98 for r=16, with the per-field breakdown showing attributes hit hardest. In live mode the gap varies run to run; what should hold is the ordering.

In [ ]:
def train_rank_variant(r, lora_alpha, splits, tokenizer, phase):
    """Train a fresh adapter at a different rank and evaluate it."""
    fresh_base, fresh_tok = ls.load_base_model()
    config = LoraConfig(
        r=r,
        lora_alpha=lora_alpha,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    variant = get_peft_model(fresh_base, config)
    variant.print_trainable_parameters()
    variant_trainer = run_training(variant, build_training_args(),
                                   splits, fresh_tok)
    return run_evaluation(variant_trainer.model, fresh_tok,
                          eval_records, phase)

In [ ]:
# Part F driver. Pre-written. Run as-is.
r4_results = None
try:
    r4_results = train_rank_variant(4, 8, ls.need(splits), tokenizer,
                                    "tuned_r4")
    ls.print_results("Part F: tuned adapter, r=4", r4_results)
except NotImplementedError:
    print("Part F not attempted (stretch goal)")

check("F1: r=4 lands at or below r=16 on schema-valid rate",
      lambda: 0 < ls.need(r4_results)["schema_valid_rate"]
              <= ls.need(tuned_results)["schema_valid_rate"])

if r4_results:
    ls.plot_before_after({"baseline": baseline_results,
                          "tuned r=4": r4_results,
                          "tuned r=16": tuned_results})

## Part G, stretch: did it forget? (20 min)

Task quality went up. The release question from Segment 4 is what went down. Run the ten-prompt capability regression set against the base and the tuned model, score both, and inspect any prompt where the tuned model degraded.

`ls.generate_capability(model, tokenizer, phase)` answers the ten generic prompts; phase is `"baseline"` or `"tuned_r16"`. `ls.score_capability(outputs)` returns a deterministic keyword score. Crude by design: a smoke alarm, not a judge. Week 6 builds the real thing.

In [ ]:
def capability_regression(base_model, trained_model, tokenizer):
    """Score general capability before and after the fine-tune."""
    base_answers = ls.generate_capability(base_model, tokenizer, "baseline")
    tuned_answers = ls.generate_capability(trained_model, tokenizer,
                                           "tuned_r16")
    return {"baseline": ls.score_capability(base_answers),
            "tuned": ls.score_capability(tuned_answers),
            "tuned_outputs": tuned_answers}

In [ ]:
# Part G driver. Pre-written. Run as-is.
cap = None
try:
    fresh_base, _ = ls.load_base_model()
    cap = capability_regression(fresh_base, ls.need(trainer).model, tokenizer)
    print(f"capability, base:  {cap['baseline']['capability_score']:.0%}")
    print(f"capability, tuned: {cap['tuned']['capability_score']:.0%}")
    for i, hit in enumerate(cap["tuned"]["per_prompt"]):
        if not hit:
            prompt = ls.CAPABILITY_PROMPTS[i][0]
            print(f"\nDEGRADED: {prompt}")
            print(f"tuned model answered: {cap['tuned_outputs'][i][:120]}")
except NotImplementedError:
    print("Part G not attempted (stretch goal)")

check("G1: capability scored for both models, tuned at or below base",
      lambda: 0 <= ls.need(cap)["tuned"]["capability_score"]
              <= cap["baseline"]["capability_score"] <= 1)

In offline mode exactly one prompt degrades, and the failure mode is worth staring at: asked a general question, the tuned model answers in the catalog schema. Format bleed. The narrow fine-tune taught format so hard that format leaks where it does not belong. This is why capability regression is a separate release gate from task quality: nothing in Part E could have caught this, because Part E only asks catalog questions.

The two-sided finding from Biderman et al. 2024 holds here in miniature: LoRA forgot little, but not nothing. Less is not none. Measure it, do not assume it.

## Wrap up

The checkpoint tally below is your progress receipt. Core lab complete means every check through E2 passes. The adapter you saved, the evaluation function you wrote, and the audit pattern from Part B all reuse directly in the capstone.

The one question to leave with, from the module: should this have been retrieval? For this task, no, and you can now defend that in one sentence. The failure mode was format, not facts. The facts were in every blurb; the model could not hold the shape. Fine-tune for behavior, retrieve for facts, combine for production.

In [ ]:
ls.checkpoint_summary()